# J&J MedTech Genie Workshop — Setup

This notebook sets up everything needed to run the J&J MedTech Genie Workshop app:
tables, metric views, a Genie space, and a Databricks App.

---

### Before you start

| Requirement | How to get it |
|---|---|
| **Unity Catalog catalog** | Must already exist. Create via Catalog Explorer if needed. |
| **SQL Warehouse ID** | SQL → Warehouses → click your warehouse → copy the ID from the URL |
| **Workshop files** | Already uploaded to the workspace at the path in `workspace_root` below |

### What this does (in order)
1. Verifies workshop files are in place
2. Creates the schema + volume, copies CSV files into the volume
3. Creates 3 Delta tables from the CSVs
4. Adds Unity Catalog metadata (comments, constraints, tags)
5. Creates 7 metric views
6. Creates or updates the Genie space
7. Uploads the app source to Workspace Files
8. Deploys and starts the Databricks App
9. Grants the app service principal the required permissions and prints the URL

**Re-run safe:** every step is idempotent — re-running updates rather than duplicates.

In [ ]:
# ── Configuration widgets ────────────────────────────────────────────────────

dbutils.widgets.text("workspace_root",  "/Workspace/Users/rleach@gmail.com/jnj-genie-workshop", "Workspace Root (folder containing src/ and raw_data/)")
dbutils.widgets.text("catalog",         "medtech",      "Catalog")
dbutils.widgets.text("schema",          "sales",        "Schema")
dbutils.widgets.text("warehouse_id",    "",             "SQL Warehouse ID")
dbutils.widgets.text("app_name",        "mt-genie-app", "App Name (lowercase, hyphens only)")
dbutils.widgets.text("genie_space_id",  "",             "Genie Space ID (blank = create new)")

workspace_root = dbutils.widgets.get("workspace_root").rstrip("/")
catalog        = dbutils.widgets.get("catalog").strip()
schema         = dbutils.widgets.get("schema").strip()
warehouse_id   = dbutils.widgets.get("warehouse_id").strip()
app_name       = dbutils.widgets.get("app_name").strip()
genie_space_id = dbutils.widgets.get("genie_space_id").strip()

DATASET     = "med_tech_sales"
VOLUME_NAME = "raw_data"

# ── Validate ─────────────────────────────────────────────────────────────────
import re

errors = []
if not catalog:     errors.append("catalog is required")
if not schema:      errors.append("schema is required")
if not warehouse_id: errors.append("warehouse_id is required")
if not app_name:    errors.append("app_name is required")
if app_name and not re.match(r'^[a-z][a-z0-9-]*$', app_name):
    errors.append("app_name must be lowercase letters, numbers, and hyphens only (e.g. mt-genie-app)")

if errors:
    for e in errors: print(f"ERROR: {e}")
    raise ValueError("Fix the widget values above and re-run.")

DATA_PATH   = f"/Volumes/{catalog}/{schema}/{VOLUME_NAME}/{DATASET}"
APP_WS_PATH = f"/Workspace/Shared/genie-workshop/{app_name}"

print(f"Workspace root: {workspace_root}")
print(f"Catalog:        {catalog}.{schema}")
print(f"Warehouse:      {warehouse_id}")
print(f"App name:       {app_name}")
print(f"Genie space:    {genie_space_id or '(will create new)'}")
print(f"Data path:      {DATA_PATH}")
print(f"App WS path:    {APP_WS_PATH}")

## Step 1 — Verify Workshop Files

In [ ]:
import os

required = [
    f"raw_data/{DATASET}/hcp_procedure_volume.csv",
    f"raw_data/{DATASET}/product_upgrades.csv",
    f"raw_data/{DATASET}/account_targeting.csv",
    f"src/notebooks/{DATASET}/01_setup_and_load.sql",
    f"src/notebooks/{DATASET}/02_add_uc_metadata.sql",
    f"src/notebooks/{DATASET}/03_add_business_semantics.sql",
    f"src/genie/{DATASET}/genie_space.json",
    f"src/app/{DATASET}/app.py",
    f"src/app/{DATASET}/app.yaml",
    f"src/app/{DATASET}/index.html",
    f"src/app/{DATASET}/requirements.txt",
]

missing = []
for rel in required:
    full = os.path.join(workspace_root, rel)
    if os.path.exists(full):
        print(f"  ✓  {rel}")
    else:
        print(f"  ✗  {rel}  ← MISSING")
        missing.append(full)

if missing:
    raise FileNotFoundError(
        f"{len(missing)} required file(s) not found under {workspace_root}.\n"
        "Make sure all workshop files have been uploaded to the workspace_root folder."
    )

print(f"\n✓ All {len(required)} files found.")

## Step 2 — Schema, Volume & CSV Files

In [ ]:
import shutil, glob

# Verify catalog exists
try:
    spark.sql(f"DESCRIBE CATALOG {catalog}")
except Exception:
    raise Exception(
        f"Catalog '{catalog}' not found.\n"
        "Create it in Catalog Explorer first, then re-run."
    )

# Create schema and volume (idempotent)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")
spark.sql(f"CREATE VOLUME  IF NOT EXISTS {catalog}.{schema}.{VOLUME_NAME}")
print(f"✓ {catalog}.{schema}.{VOLUME_NAME} ready")

# Copy CSVs from workspace into the volume
os.makedirs(DATA_PATH, exist_ok=True)
csv_src   = os.path.join(workspace_root, "raw_data", DATASET)
csv_files = glob.glob(os.path.join(csv_src, "*.csv"))

if not csv_files:
    raise FileNotFoundError(f"No CSV files found at {csv_src}")

for src in csv_files:
    dst = os.path.join(DATA_PATH, os.path.basename(src))
    shutil.copy2(src, dst)

files = dbutils.fs.ls(f"dbfs:{DATA_PATH}")
print(f"✓ {len(files)} CSV files in volume:")
for f in files:
    print(f"    {f.name:<45} {f.size:>10,} bytes")

## Step 3 — Create Delta Tables

In [ ]:
def run_sql_file(path, catalog, schema, volume_name):
    """Read a SQL notebook file, substitute params, and run each statement."""
    with open(path, 'r') as f:
        content = f.read()

    content = (content
               .replace('${catalog}',     catalog)
               .replace('${schema}',      schema)
               .replace('${volume_name}', volume_name))

    # Strip notebook magic lines and cell separators
    content = re.sub(r'-- MAGIC.*\n', '\n', content)
    content = re.sub(r'-- Databricks notebook source\n', '', content)
    content = re.sub(r'-- COMMAND ----------\n?', '\n', content)

    # Split on statement-ending semicolons
    statements = re.split(r';\s*\n', content)

    ok = skipped = 0
    for raw in statements:
        stmt = raw.strip()
        lines = [l.strip() for l in stmt.splitlines() if l.strip()]
        if not lines or all(l.startswith('--') for l in lines):
            continue
        try:
            spark.sql(stmt)
            ok += 1
            print(f"  ✓  {stmt.splitlines()[0][:80]}")
        except Exception as e:
            if any(x in str(e).lower() for x in ('already exists', 'duplicate', 'constraint')):
                skipped += 1
                print(f"  ~  {stmt.splitlines()[0][:80]}  (already exists, skipped)")
            else:
                print(f"  ✗  {stmt.splitlines()[0][:80]}")
                raise
    print(f"\n  {ok} run, {skipped} skipped")

sql_01 = os.path.join(workspace_root, "src", "notebooks", DATASET, "01_setup_and_load.sql")
run_sql_file(sql_01, catalog, schema, VOLUME_NAME)
print("✓ Tables created")

## Step 4 — Add Unity Catalog Metadata

In [ ]:
sql_02 = os.path.join(workspace_root, "src", "notebooks", DATASET, "02_add_uc_metadata.sql")
run_sql_file(sql_02, catalog, schema, VOLUME_NAME)
print("✓ UC metadata applied")

## Step 5 — Create Metric Views

In [ ]:
sql_03 = os.path.join(workspace_root, "src", "notebooks", DATASET, "03_add_business_semantics.sql")
run_sql_file(sql_03, catalog, schema, VOLUME_NAME)
print("✓ Metric views created")

## Step 6 — Create / Update Genie Space

In [ ]:
import json, requests, time
from databricks.sdk import WorkspaceClient

w       = WorkspaceClient()
host    = w.config.host.rstrip("/")
headers = dict(w.config.authenticate())
headers["Content-Type"] = "application/json"

genie_json_path = os.path.join(workspace_root, "src", "genie", DATASET, "genie_space.json")
with open(genie_json_path, 'r') as f:
    raw = f.read()

raw = re.sub(r'__CATALOG__\.__SCHEMA__', f'{catalog}.{schema}', raw)
raw = raw.replace('__WAREHOUSE_ID__',  warehouse_id)
raw = raw.replace('__GENIE_SPACE_ID__', genie_space_id or '')

genie_cfg            = json.loads(raw)
genie_cfg["warehouse_id"] = warehouse_id
if genie_space_id:
    genie_cfg["space_id"] = genie_space_id

serialized = json.dumps(genie_cfg.get("serialized_space", ""))
payload = {
    "warehouse_id":     warehouse_id,
    "serialized_space": serialized,
    "title":            genie_cfg["display_name"],
    "description":      genie_cfg["description"],
}

final_space_id = None

if genie_space_id:
    print(f"Updating Genie space {genie_space_id} ...")
    resp = requests.patch(f"{host}/api/2.0/genie/spaces/{genie_space_id}", headers=headers, json=payload)
    if resp.status_code == 200:
        final_space_id = genie_space_id
        print(f"✓ Updated: {final_space_id}")
    else:
        print(f"  ({resp.status_code}) — will create a new space instead.")

if not final_space_id:
    print("Creating new Genie space ...")
    resp = requests.post(f"{host}/api/2.0/genie/spaces", headers=headers, json=payload)
    if resp.status_code not in (200, 201):
        raise Exception(f"Failed ({resp.status_code}): {resp.text[:400]}")
    data           = resp.json()
    final_space_id = data.get("space_id") or data.get("id")
    print(f"✓ Created: {final_space_id}")

print(f"\n{'='*60}")
print(f"  GENIE SPACE ID: {final_space_id}")
print(f"  Paste this into the genie_space_id widget on re-runs.")
print(f"{'='*60}")

## Step 7 — Upload App Source to Workspace

In [ ]:
import base64

def upload_workspace_file(ws_path, content_bytes):
    h = {**headers}
    parent = ws_path.rsplit("/", 1)[0]
    requests.post(f"{host}/api/2.0/workspace/mkdirs", headers=h, json={"path": parent})
    resp = requests.post(
        f"{host}/api/2.0/workspace/import", headers=h,
        json={
            "path":      ws_path,
            "format":    "AUTO",
            "content":   base64.b64encode(content_bytes).decode(),
            "overwrite": True,
        }
    )
    return resp

requests.post(f"{host}/api/2.0/workspace/mkdirs", headers=headers, json={"path": APP_WS_PATH})

app_src = os.path.join(workspace_root, "src", "app", DATASET)
for fname in ["app.py", "app.yaml", "index.html", "requirements.txt"]:
    fpath = os.path.join(app_src, fname)
    with open(fpath, 'rb') as f:
        content = f.read()
    if fname == "app.yaml":
        content = (content
                   .replace(b'__WAREHOUSE_ID__',   warehouse_id.encode())
                   .replace(b'__GENIE_SPACE_ID__',  final_space_id.encode()))
    resp = upload_workspace_file(f"{APP_WS_PATH}/{fname}", content)
    status = "✓" if resp.status_code == 200 else f"✗ ({resp.status_code}) {resp.text[:80]}"
    print(f"  {status}  {fname}")

print(f"\n✓ App source uploaded to {APP_WS_PATH}")

## Step 8 — Deploy & Start App

In [ ]:
resp = requests.get(f"{host}/api/2.0/apps/{app_name}", headers=headers)
if resp.status_code == 404:
    print(f"Creating app '{app_name}' ...")
    resp = requests.post(f"{host}/api/2.0/apps", headers=headers,
                         json={"name": app_name, "description": "MedTech Sales Genie Workshop App"})
    resp.raise_for_status()
    print("✓ App created")
    time.sleep(3)
elif resp.status_code == 200:
    print(f"App '{app_name}' exists — redeploying.")
else:
    resp.raise_for_status()

print(f"Deploying from {APP_WS_PATH} ...")
resp = requests.post(
    f"{host}/api/2.0/apps/{app_name}/deployments", headers=headers,
    json={"source_code_path": APP_WS_PATH, "mode": "SNAPSHOT"}
)
resp.raise_for_status()
deployment_id = resp.json().get("deployment_id") or resp.json().get("id")
print(f"✓ Deployment started: {deployment_id}")

for i in range(30):
    time.sleep(5)
    dr    = requests.get(f"{host}/api/2.0/apps/{app_name}/deployments/{deployment_id}", headers=headers).json()
    state = (dr.get("status") or {}).get("state", dr.get("state", "PENDING"))
    msg   = (dr.get("status") or {}).get("message", "")
    print(f"  [{i+1}/30] {state}  {msg[:80]}")
    if state == "SUCCEEDED":
        print("✓ Deployment complete")
        break
    if state in ("FAILED", "CANCELLED"):
        raise Exception(f"Deployment {state}: {dr}")

resp = requests.post(f"{host}/api/2.0/apps/{app_name}/start", headers=headers)
print(f"✓ App start requested ({resp.status_code})")

## Step 9 — Grant Permissions & Get URL

In [ ]:
app_data    = requests.get(f"{host}/api/2.0/apps/{app_name}", headers=headers).json()
sp_client_id = app_data.get("service_principal_client_id", "")

if sp_client_id:
    print(f"App SP client_id: {sp_client_id}")

    resp = requests.patch(
        f"{host}/api/2.0/permissions/genie/{final_space_id}", headers=headers,
        json={"access_control_list": [{"service_principal_name": sp_client_id, "permission_level": "CAN_MANAGE"}]}
    )
    print(f"  Genie CAN_MANAGE:  {'✓' if resp.status_code == 200 else f'✗ ({resp.status_code})'}")

    resp = requests.patch(
        f"{host}/api/2.0/permissions/warehouses/{warehouse_id}", headers=headers,
        json={"access_control_list": [{"service_principal_name": sp_client_id, "permission_level": "CAN_USE"}]}
    )
    print(f"  Warehouse CAN_USE: {'✓' if resp.status_code == 200 else f'✗ ({resp.status_code})'}")

    for sql in [
        f"GRANT USE_CATALOG ON CATALOG {catalog} TO `{sp_client_id}`",
        f"GRANT USE_SCHEMA   ON SCHEMA  {catalog}.{schema} TO `{sp_client_id}`",
        f"GRANT SELECT       ON SCHEMA  {catalog}.{schema} TO `{sp_client_id}`",
    ]:
        try:
            spark.sql(sql)
            print(f"  ✓  {sql}")
        except Exception as e:
            print(f"  ~  {sql}  ({e})")
else:
    print("WARNING: App SP not found — permissions not granted. Re-run this cell once the app is running.")

# Wait for RUNNING
print("\nWaiting for app to reach RUNNING state ...")
app_url = None
for i in range(30):
    time.sleep(10)
    data    = requests.get(f"{host}/api/2.0/apps/{app_name}", headers=headers).json()
    state   = (data.get("app_status") or data.get("status") or {}).get("state", "")
    message = (data.get("app_status") or data.get("status") or {}).get("message", "")
    app_url = data.get("url", "")
    print(f"  [{i+1}/30] {state:<12}  {message[:80]}")
    if state == "RUNNING":
        break
    if state in ("CRASHED", "STOPPED"):
        print(f"  App {state}. Check logs: Databricks → Apps → {app_name} → Logs")
        break

print()
if app_url:
    displayHTML(f"""
    <div style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Roboto,sans-serif;
                padding:28px 32px;background:#fff;border:1px solid #e0e0e0;
                border-radius:12px;max-width:600px;margin:16px 0">
      <div style="color:#EB1700;font-size:22px;font-weight:700;margin-bottom:8px">&#9989; Setup Complete</div>
      <div style="color:#333;font-size:15px;margin-bottom:20px">Your J&amp;J MedTech Genie Workshop app is live.</div>
      <div style="margin-bottom:16px">
        <span style="font-size:12px;color:#888;text-transform:uppercase;letter-spacing:.5px">App URL</span><br>
        <a href="{app_url}" target="_blank" style="font-size:16px;color:#1565C0">{app_url}</a>
      </div>
      <div style="margin-bottom:16px">
        <span style="font-size:12px;color:#888;text-transform:uppercase;letter-spacing:.5px">Genie Space ID</span><br>
        <code style="font-size:14px;color:#333">{final_space_id}</code>
      </div>
      <div style="font-size:12px;color:#888;border-top:1px solid #eee;padding-top:12px">
        Save the Genie Space ID — paste it into the <code>genie_space_id</code> widget on re-runs to update rather than recreate.
      </div>
    </div>
    """)
else:
    print(f"App deploying. Check: {host}/apps/{app_name}")
    print(f"Genie Space ID: {final_space_id}")